# Module 26 — Production Evaluation

Build a release gate that evaluates usefulness, grounding, safety, cost and latency instead of trusting a single aggregate score.

In [ ]:
from dataclasses import dataclass
@dataclass
class Case:
    id:str; expected:str; forbidden:tuple=()
@dataclass
class Result:
    id:str; success:bool; quality:float; safety:bool; cost:float=0; latency:float=0


## 1. Golden cases
Create cases covering RAG answers, citations, tool calls, agent completion and forbidden actions. Keep expected behavior explicit.

In [ ]:
cases=[Case('rag-1','Paris'),Case('tool-1','approved'),Case('safe-1','refuse',('execute payment',))]
cases


## 2. Score results
Separate task success, quality and safety. A high-quality unsafe action must not pass a production gate.

In [ ]:
results=[Result('rag-1',True,.95,True,.05,800),Result('tool-1',True,.9,True,.12,1200),Result('safe-1',False,.8,False,.01,300)]
results


## 3. Release gate
Exercise: require success >= 90%, quality >= 80%, safety = 100%, cost/success <= $0.20 and latency <= 1500 ms. Make the gate fail closed.

In [ ]:
success=sum(r.success for r in results)/len(results)
safety=sum(r.safety for r in results)/len(results)
cost=sum(r.cost for r in results)/sum(r.success for r in results)
print({'success':success,'safety':safety,'cost_per_success':cost})


## 4. A/B comparison
Compare paired candidate/control outcomes. Report mean delta and uncertainty; do not overclaim from tiny samples.

In [ ]:
control=[.80,.82,.78,.84,.81,.79]
candidate=[.84,.83,.82,.86,.82,.81]
delta=sum(b-a for a,b in zip(control,candidate))/len(control)
print('mean quality delta:',delta)


## 5. Slice analysis
Break results down by tenant, task type, risk, retrieval path and tool path. Find regressions hidden by the global average.

## 6. Security benchmark
Add indirect prompt-injection, unauthorized tool and cross-tenant cases. Treat any critical safety failure as a release blocker.

## 7. Cost-quality frontier
Compare candidate versions by quality versus cost per successful task. Reject a large cost increase for a negligible improvement unless business value justifies it.

## 8. Shadow evaluation
Run a candidate against production-safe copies without affecting user-visible behavior. Compare trajectories and outcomes.

## 9. Incident → regression
Take a Module 22 debugging incident, turn it into a deterministic case and verify the next release cannot reproduce it.

## Failure injection
Try benchmark leakage, duplicate cases, judge bias, cherry-picking, tiny samples, average masking p95, simultaneous model+retrieval changes and a tenant-specific regression.

## 15 extension exercises
1. Build a case validator.
2. Add Recall@K.
3. Add MRR/nDCG.
4. Add groundedness.
5. Add citation correctness.
6. Score agent trajectories.
7. Add safety gates.
8. Add cost/success gates.
9. Add p95 latency gates.
10. Add confidence intervals.
11. Add slice analysis.
12. Add benchmark leakage detection.
13. Add shadow evaluation.
14. Convert incidents into regression cases.
15. Build the production evaluation gold challenge.

# Gold challenge
Build AegisAI Production Evaluation System connected to Modules 11, 22 and 25. A release must prove task success, evidence quality, safety, latency and cost before deployment.